In [1]:

from  google.colab import files
uploaded=files.upload()

Saving online_retail_II.xlsx to online_retail_II.xlsx


# Task 3 — Data Cleaning

## Objective
Clean and transform a deliberately messy dataset into a reliable, analysis-ready dataset using Python, pandas, and numpy.

The cleaning process will systematically address missing values, duplicate records, inconsistent formatting, outliers, and incorrect data types. Each major cleaning decision will be documented and validated.

In [2]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_excel('online_retail_II.xlsx')

In [8]:
#Initial Inspection

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Dataset shape: (525461, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 5 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [9]:
# Initial Data-Quality Report

quality_report = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2),
    "Unique Values": df.nunique()
})

print("Duplicate rows:", df.duplicated().sum())

display(quality_report)

Duplicate rows: 6865


,Data Type,Missing Values,Missing %,Unique Values
Invoice,object,0,0.00,28816
StockCode,object,0,0.00,4632
Description,object,2928,0.56,4681
Quantity,int64,0,0.00,825
InvoiceDate,datetime64[ns],0,0.00,25296
Price,float64,0,0.00,1606
Customer ID,float64,107927,20.54,4383
Country,object,0,0.00,40


In [10]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice,525461.0,28816.0,537434.0,675.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,525461,4632,85123A,3516,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,522533,4681,WHITE HANGING HEART T-LIGHT HOLDER,3549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,525461.0,NaN,NaN,NaN,10.337667,-9600.0,1.0,3.0,10.0,19152.0,107.42411
InvoiceDate,525461,NaN,NaN,NaN,2010-06-28 11:37:36.845017856,2009-12-01 07:45:00,2010-03-21 12:20:00,2010-07-06 09:51:00,2010-10-15 12:45:00,2010-12-09 20:01:00,NaN
Price,525461.0,NaN,NaN,NaN,4.688834,-53594.36,1.25,2.1,4.21,25111.09,146.126914
Customer ID,417534.0,NaN,NaN,NaN,15360.645478,12346.0,13983.0,15311.0,16799.0,18287.0,1680.811316
Country,525461,40,United Kingdom,485852,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
categorical_columns = df.select_dtypes(include=["object", "category"]).columns

for col in categorical_columns:
    print(f"\n{'='*60}")
    print(f"Column: {col}")
    print(f"Unique values: {df[col].nunique()}")
    print(df[col].value_counts(dropna=False).head(20))


Column: Invoice
Unique values: 28816
Invoice
537434    675
538071    652
537638    601
537237    597
536876    593
536592    592
537823    591
536031    582
490074    580
491966    579
537240    568
490149    559
491969    548
490741    546
537666    536
536544    527
489857    516
513574    515
490745    507
489597    502
Name: count, dtype: int64

Column: StockCode
Unique values: 4632
StockCode
85123A    3516
22423     2221
85099B    2057
21212     1933
21232     1843
20725     1620
84879     1458
84991     1400
21754     1386
20914     1276
21034     1232
21931     1220
21080     1209
22139     1203
21977     1196
22383     1192
22138     1180
20727     1179
82494L    1165
22470     1154
Name: count, dtype: int64

Column: Description
Unique values: 4681
Description
WHITE HANGING HEART T-LIGHT HOLDER    3549
NaN                                   2928
REGENCY CAKESTAND 3 TIER              2212
STRAWBERRY CERAMIC TRINKET BOX        1843
PACK OF 72 RETRO SPOT CAKE CASES      1466
ASSOR

In [12]:
# Numeric Range Check

numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    print(f"\n{col}")
    print(f"Min: {df[col].min()}")
    print(f"Max: {df[col].max()}")
    print(f"Mean: {df[col].mean():.2f}")
    print(f"Median: {df[col].median():.2f}")


Quantity
Min: -9600
Max: 19152
Mean: 10.34
Median: 3.00

Price
Min: -53594.36
Max: 25111.09
Mean: 4.69
Median: 2.10

Customer ID
Min: 12346.0
Max: 18287.0
Mean: 15360.65
Median: 15311.00


In [13]:

# Diagnostic checks for suspicious and potentially invalid data

# 1. Zero and negative values
print("QUANTITY CHECK")
print("-" * 50)
print("Zero Quantity:", (df["Quantity"] == 0).sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Positive Quantity:", (df["Quantity"] > 0).sum())

print("\nPRICE CHECK")
print("-" * 50)
print("Zero Price:", (df["Price"] == 0).sum())
print("Negative Price:", (df["Price"] < 0).sum())
print("Positive Price:", (df["Price"] > 0).sum())


# 2. Negative quantity records
negative_quantity = df[df["Quantity"] < 0]

print("\nNEGATIVE QUANTITY RECORDS")
print("-" * 50)
print("Rows:", len(negative_quantity))

display(
    negative_quantity[
        ["Invoice", "StockCode", "Description", "Quantity",
         "InvoiceDate", "Price", "Customer ID", "Country"]
    ].head(10)
)


# 3. Invoice patterns associated with negative quantities
print("\nINVOICE PATTERNS")
print("-" * 50)

invoice_string = df["Invoice"].astype(str)

print("Invoices starting with 'C':", invoice_string.str.startswith("C").sum())
print("Negative quantities with 'C' invoice:",
      negative_quantity["Invoice"].astype(str).str.startswith("C").sum())


# 4. Negative prices
negative_price = df[df["Price"] < 0]

print("\nNEGATIVE PRICE RECORDS")
print("-" * 50)
print("Rows:", len(negative_price))

display(
    negative_price[
        ["Invoice", "StockCode", "Description", "Quantity",
         "InvoiceDate", "Price", "Customer ID", "Country"]
    ].head(10)
)


# 5. Zero-price records
zero_price = df[df["Price"] == 0]

print("\nZERO PRICE RECORDS")
print("-" * 50)
print("Rows:", len(zero_price))

display(
    zero_price[
        ["Invoice", "StockCode", "Description", "Quantity",
         "InvoiceDate", "Price", "Customer ID", "Country"]
    ].head(10)
)


# 6. Missing Customer IDs
missing_customer = df[df["Customer ID"].isna()]

print("\nMISSING CUSTOMER ID")
print("-" * 50)
print("Rows:", len(missing_customer))
print("Unique invoices:", missing_customer["Invoice"].nunique())
print("Unique stock codes:", missing_customer["StockCode"].nunique())

display(missing_customer.head(10))


# 7. Missing descriptions
missing_description = df[df["Description"].isna()]

print("\nMISSING DESCRIPTION")
print("-" * 50)
print("Rows:", len(missing_description))
print("Unique invoices:", missing_description["Invoice"].nunique())
print("Unique stock codes:", missing_description["StockCode"].nunique())

display(missing_description.head(10))


# 8. Check for whitespace in text columns
print("\nTEXT FORMATTING CHECK")
print("-" * 50)

for col in ["Invoice", "StockCode", "Description", "Country"]:
    whitespace_count = (
        df[col].astype("string").str.strip().ne(df[col].astype("string"))
    ).sum()

    print(f"{col}: {whitespace_count} values with leading/trailing whitespace")

QUANTITY CHECK
--------------------------------------------------
Zero Quantity: 0
Negative Quantity: 12326
Positive Quantity: 513135

PRICE CHECK
--------------------------------------------------
Zero Price: 3687
Negative Price: 3
Positive Price: 521771

NEGATIVE QUANTITY RECORDS
--------------------------------------------------
Rows: 12326


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom



INVOICE PATTERNS
--------------------------------------------------
Invoices starting with 'C': 10206
Negative quantities with 'C' invoice: 10205

NEGATIVE PRICE RECORDS
--------------------------------------------------
Rows: 3


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom



ZERO PRICE RECORDS
--------------------------------------------------
Rows: 3687


...


MISSING CUSTOMER ID
--------------------------------------------------
Rows: 107927
Unique invoices: 5229
Unique stock codes: 4407


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom



MISSING DESCRIPTION
--------------------------------------------------
Rows: 2928
Unique invoices: 2928
Unique stock codes: 1920


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom



TEXT FORMATTING CHECK
--------------------------------------------------
Invoice: 0 values with leading/trailing whitespace
StockCode: 1 values with leading/trailing whitespace
Description: 99583 values with leading/trailing whitespace
Country: 0 values with leading/trailing whitespace


In [14]:

# Investigating whether missing values can be safely recovered

# 1. Customer ID: check whether missing IDs occur within
#    otherwise identified invoices

invoice_customer_summary = (
    df.groupby("Invoice")["Customer ID"]
      .agg(
          total_rows="size",
          known_customer_ids=lambda x: x.notna().sum(),
          missing_customer_ids=lambda x: x.isna().sum(),
          unique_known_customer_ids=lambda x: x.dropna().nunique()
      )
)

print("CUSTOMER ID COMPLETENESS BY INVOICE")
print("-" * 60)

print(
    "Invoices with at least one known Customer ID:",
    (invoice_customer_summary["known_customer_ids"] > 0).sum()
)

print(
    "Invoices with some missing Customer IDs:",
    (invoice_customer_summary["missing_customer_ids"] > 0).sum()
)

print(
    "Invoices with ALL Customer IDs missing:",
    (invoice_customer_summary["known_customer_ids"] == 0).sum()
)

print(
    "Invoices with multiple known Customer IDs:",
    (invoice_customer_summary["unique_known_customer_ids"] > 1).sum()
)

# 2. Checking how many missing Customer ID rows could potentially
#    be filled from another row within the same invoice


invoice_has_known_customer = (
    df.groupby("Invoice")["Customer ID"]
      .transform("count") > 0
)

fillable_customer_rows = (
    df["Customer ID"].isna() & invoice_has_known_customer
)

print(
    "\nMissing Customer ID rows that could potentially be "
    "filled from the same invoice:",
    fillable_customer_rows.sum()
)

print(
    "Missing Customer ID rows in invoices where the entire "
    "invoice has no Customer ID:",
    (
        df["Customer ID"].isna() &
        ~invoice_has_known_customer
    ).sum()
)

# 3. Inspecting invoices containing both known and missing IDs

mixed_customer_invoices = invoice_customer_summary[
    (invoice_customer_summary["known_customer_ids"] > 0) &
    (invoice_customer_summary["missing_customer_ids"] > 0)
].index

print("\nExample invoices with mixed Customer ID availability:")
print(mixed_customer_invoices[:10].tolist())

display(
    df[
        df["Invoice"].isin(mixed_customer_invoices[:3])
    ][
        ["Invoice", "StockCode", "Description",
         "Quantity", "Customer ID", "Country"]
    ].head(30)
)

# 4. Description recovery check
#    Can missing descriptions be recovered using StockCode?

stockcode_description = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .nunique()
)

missing_description_stockcodes = (
    df.loc[df["Description"].isna(), "StockCode"]
      .unique()
)

recoverable_description_stockcodes = [
    code for code in missing_description_stockcodes
    if stockcode_description.get(code, 0) == 1
]

print("\nDESCRIPTION RECOVERY")
print("-" * 60)

print(
    "Unique StockCodes associated with missing descriptions:",
    len(missing_description_stockcodes)
)

print(
    "StockCodes with exactly one known description:",
    len(recoverable_description_stockcodes)
)

print(
    "Missing-description rows potentially recoverable:",
    df.loc[
        df["Description"].isna() &
        df["StockCode"].isin(recoverable_description_stockcodes)
    ].shape[0]
)

# 5. Checking zero-price records more closely
print("\nZERO PRICE TRANSACTION TYPES")
print("-" * 60)

display(
    df[df["Price"] == 0][
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Customer ID", "Country"]
    ].head(20)
)

CUSTOMER ID COMPLETENESS BY INVOICE
------------------------------------------------------------
Invoices with at least one known Customer ID: 23587
Invoices with some missing Customer IDs: 5229
Invoices with ALL Customer IDs missing: 5229
Invoices with multiple known Customer IDs: 0

Missing Customer ID rows that could potentially be filled from the same invoice: 0
Missing Customer ID rows in invoices where the entire invoice has no Customer ID: 107927

Example invoices with mixed Customer ID availability:
[]


,Invoice,StockCode,Description,Quantity,Customer ID,Country



DESCRIPTION RECOVERY
------------------------------------------------------------
Unique StockCodes associated with missing descriptions: 1920
StockCodes with exactly one known description: 1331
Missing-description rows potentially recoverable: 2150

ZERO PRICE TRANSACTION TYPES
------------------------------------------------------------


,Invoice,StockCode,Description,Quantity,InvoiceDate,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,NaN,United Kingdom


## Missing Data Handling Strategy

Missing values were handled based on the meaning of each column and the evidence found during the diagnostic analysis.

### Customer ID
Customer ID is missing for 107,927 rows (20.54% of the dataset). These missing values occur across 5,229 invoices, and every affected invoice has the Customer ID missing for all of its rows. Therefore, the values cannot be reliably recovered from other rows within the same invoice.

**Decision:** Remove rows with missing Customer ID.

**Reason:** Customer ID is important for customer-level transaction analysis. Imputing an identifier using the mean, median, or mode would create false customer identities, while forward filling would assign unrelated customers. Row deletion is therefore the safest option for producing a reliable customer-level dataset.

### Description
There are 2,928 missing product descriptions. A StockCode-based investigation found that 2,150 of these rows can be reliably recovered because their StockCode has exactly one known description elsewhere in the dataset.

**Decision:** Recover descriptions using the unique StockCode-to-Description relationship. Remove the remaining unrecoverable rows.

**Reason:** This preserves valid transaction records where the product description can be confidently identified while avoiding arbitrary text imputation.

### Other columns
Invoice, StockCode, Quantity, InvoiceDate, Price, and Country do not contain missing values and therefore do not require missing-value imputation.

In [15]:

# Recording the dataset state before cleaning

rows_before = len(df)
columns_before = df.shape[1]

missing_before = df.isnull().sum().sum()
duplicate_before = df.duplicated().sum()

print("BEFORE CLEANING")
print("-" * 50)
print("Rows:", rows_before)
print("Columns:", columns_before)
print("Total missing values:", missing_before)
print("Duplicate rows:", duplicate_before)

BEFORE CLEANING
--------------------------------------------------
Rows: 525461
Columns: 8
Total missing values: 110855
Duplicate rows: 6865


In [16]:

# Handling missing product descriptions


# Creating a mapping only for StockCodes with exactly one
# known description.
description_mapping = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .agg(lambda x: x.iloc[0] if x.nunique() == 1 else np.nan)
      .dropna()
)

# Filling missing descriptions where the StockCode has a
# unique and reliable description.
missing_description_mask = df["Description"].isna()

df.loc[missing_description_mask, "Description"] = (
    df.loc[missing_description_mask, "StockCode"]
      .map(description_mapping)
)

# Counting descriptions that remain missing after recovery.
remaining_missing_description = df["Description"].isna().sum()

print("Descriptions recovered:",
      2928 - remaining_missing_description)

print("Descriptions still missing:",
      remaining_missing_description)

# Removign rows where the description could not be reliably recovered.
df = df.dropna(subset=["Description"]).copy()

print("\nRows after handling Description:",
      len(df))

Descriptions recovered: 2150
Descriptions still missing: 778

Rows after handling Description: 524683


In [17]:

# Handling missing Customer IDs


missing_customer_before = df["Customer ID"].isna().sum()

# Removing transactions without a reliable Customer ID.
df = df.dropna(subset=["Customer ID"]).copy()

missing_customer_after = df["Customer ID"].isna().sum()

print("Missing Customer IDs before:", missing_customer_before)
print("Rows removed:", missing_customer_before - missing_customer_after)
print("Missing Customer IDs after:", missing_customer_after)

print("\nDataset shape after missing-value handling:",
      df.shape)

Missing Customer IDs before: 107149
Rows removed: 107149
Missing Customer IDs after: 0

Dataset shape after missing-value handling: (417534, 8)


In [18]:

# Removing exact duplicate rows

duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().copy()

duplicates_after = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)
print("Duplicate rows removed:", duplicates_before - duplicates_after)
print("Duplicate rows after removal:", duplicates_after)

print("\nDataset shape after duplicate removal:", df.shape)

Duplicate rows before removal: 6771
Duplicate rows removed: 6771
Duplicate rows after removal: 0

Dataset shape after duplicate removal: (410763, 8)


## Duplicate Removal

Exact duplicate rows were identified using all columns and removed using `drop_duplicates()`.

Duplicate removal was performed after missing-value handling so that the cleaning process operates on the records that will remain in the final analysis-ready dataset.

Only exact duplicate records were removed. Records that share individual values but represent different transactions were retained.

In [19]:

# Standardizing text formatting


text_columns = ["Invoice", "StockCode", "Description", "Country"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

print("Text standardization completed.")

print("\nRemaining leading/trailing whitespace:")
for col in text_columns:
    whitespace_count = (
        df[col].str.strip().ne(df[col])
    ).sum()

    print(f"{col}: {whitespace_count}")

Text standardization completed.

Remaining leading/trailing whitespace:
Invoice: 0
StockCode: 0
Description: 0
Country: 0


In [20]:

# Correcting Customer ID data type

df["Customer ID"] = df["Customer ID"].astype(int).astype(str)

print("Updated data types:")
print(df.dtypes)

Updated data types:
Invoice        string[python]
StockCode      string[python]
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID            object
Country        string[python]
dtype: object


In [21]:

# Removing invalid negative-price records


negative_price_before = (df["Price"] < 0).sum()

df = df[df["Price"] >= 0].copy()

negative_price_after = (df["Price"] < 0).sum()

print("Negative-price records before:", negative_price_before)
print("Negative-price records removed:",
      negative_price_before - negative_price_after)
print("Negative-price records after:", negative_price_after)

print("\nDataset shape:", df.shape)

Negative-price records before: 0
Negative-price records removed: 0
Negative-price records after: 0

Dataset shape: (410763, 8)


## Handling Negative Prices

Three records contained negative prices. Investigation showed that all three were labelled `Adjust bad debt` rather than representing normal product transactions.

Because `Price` represents the monetary price of a product, these records would distort transaction-level price analysis. They were therefore removed.

Negative quantities were **not** removed because the diagnostic analysis showed that they primarily correspond to cancellation/return transactions and therefore represent meaningful business events.

In [22]:

# Outlier Detection Using the IQR Method


def iqr_outlier_summary(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    )

    return {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outliers": outlier_mask.sum(),
        "Outlier %": round(outlier_mask.mean() * 100, 2)
    }


quantity_outliers = iqr_outlier_summary(df, "Quantity")
price_outliers = iqr_outlier_summary(df, "Price")

outlier_summary = pd.DataFrame(
    [quantity_outliers, price_outliers],
    index=["Quantity", "Price"]
)

display(outlier_summary)

,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers,Outlier %
Quantity,2.00,12.00,10.0,-13.0,27.0,27342,6.66
Price,1.25,3.75,2.5,-2.5,7.5,34703,8.45


In [23]:

# Inspecting IQR-detected outliers


def get_iqr_bounds(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return lower, upper


quantity_lower, quantity_upper = get_iqr_bounds(df, "Quantity")
price_lower, price_upper = get_iqr_bounds(df, "Price")


quantity_outlier_mask = (
    (df["Quantity"] < quantity_lower) |
    (df["Quantity"] > quantity_upper)
)

price_outlier_mask = (
    (df["Price"] < price_lower) |
    (df["Price"] > price_upper)
)


print("QUANTITY OUTLIERS")
print("-" * 60)
print("Lower bound:", quantity_lower)
print("Upper bound:", quantity_upper)
print("Outlier rows:", quantity_outlier_mask.sum())

display(
    df.loc[
        quantity_outlier_mask,
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price",
         "Customer ID", "Country"]
    ]
    .sort_values("Quantity")
    .head(15)
)


print("\nPRICE OUTLIERS")
print("-" * 60)
print("Lower bound:", price_lower)
print("Upper bound:", price_upper)
print("Outlier rows:", price_outlier_mask.sum())

display(
    df.loc[
        price_outlier_mask,
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price",
         "Customer ID", "Country"]
    ]
    .sort_values("Price")
    .head(15)
)

QUANTITY OUTLIERS
------------------------------------------------------------
Lower bound: -13.0
Upper bound: 27.0
Outlier rows: 27342


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838,United Kingdom
359669,C524235,21088,SET/6 FRUIT SALAD PAPER CUPS,-7128,2010-09-28 11:02:00,0.08,14277,France
359670,C524235,21096,SET/6 FRUIT SALAD PAPER PLATES,-7008,2010-09-28 11:02:00,0.13,14277,France
359630,C524235,16047,POP ART PEN CASE & PENS,-5184,2010-09-28 11:02:00,0.08,14277,France
359636,C524235,37340,MULTICOLOUR SPRING FLOWER MUG,-4992,2010-09-28 11:02:00,0.10,14277,France
359653,C524235,85110,BLACK SILVER FLOWER T-LIGHT HOLDER,-4752,2010-09-28 11:02:00,0.07,14277,France
359658,C524235,16046,TEATIME PEN CASE & PENS,-4608,2010-09-28 11:02:00,0.08,14277,France
359654,C524235,85160A,WHITE BIRD GARDEN DESIGN MUG,-4320,2010-09-28 11:02:00,0.13,14277,France
359674,C524235,85184D,S/4 BLUE ROUND DECOUPAGE BOXES,-3936,2010-09-28 11:02:00,0.42,14277,France
359660,C524235,16162L,THE KING GIFT BAG,-3744,2010-09-28 11:02:00,0.05,14277,France



PRICE OUTLIERS
------------------------------------------------------------
Lower bound: -2.5
Upper bound: 7.5
Outlier rows: 34703


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
129377,C501692,D,Discount,-1,2010-03-18 17:26:00,7.63,13798,United Kingdom
64900,495185,22172,METAL SHELF WITH RAIL,8,2010-01-21 14:01:00,7.65,14439,Greece
406183,528311,22941,CHRISTMAS LIGHTS 10 REINDEER,24,2010-10-21 12:17:00,7.65,17924,United Kingdom
406174,528310,22942,CHRISTMAS LIGHTS 10 SANTAS,24,2010-10-21 12:14:00,7.65,14156,EIRE
281204,516769,22636,CHILDS BREAKFAST SET CIRCUS PARADE,8,2010-07-22 17:33:00,7.65,13371,United Kingdom
407128,528375,22942,CHRISTMAS LIGHTS 10 SANTAS,48,2010-10-21 17:29:00,7.65,13093,United Kingdom
407126,528375,22941,CHRISTMAS LIGHTS 10 REINDEER,72,2010-10-21 17:29:00,7.65,13093,United Kingdom
281143,C516764,22215,CAKE STAND WHITE TWO TIER LACE,-12,2010-07-22 17:05:00,7.65,14199,United Kingdom
407602,528388,22942,CHRISTMAS LIGHTS 10 SANTAS,48,2010-10-21 18:37:00,7.65,13093,United Kingdom
407600,528388,22941,CHRISTMAS LIGHTS 10 REINDEER,72,2010-10-21 18:37:00,7.65,13093,United Kingdom


## Outlier Detection and Decision

The IQR method was applied to the numeric `Quantity` and `Price` columns.

### Quantity

The IQR method identified 27,342 potential outliers (6.66% of the dataset), with values outside the range of -13 to 27.

Inspection showed that many extreme values are legitimate business transactions. In particular, large negative quantities are associated with cancellation/return invoices beginning with `C`. Large positive quantities can also represent legitimate bulk purchases.

**Decision: Retain Quantity outliers.**

Removing or capping these values would alter legitimate transaction quantities and could distort sales and return analysis.

### Price

The IQR method identified 34,703 potential outliers (8.45%), with prices above 7.50 classified as statistical outliers.

Inspection showed that many of these values are valid product prices. The relatively low IQR upper boundary results from the strong concentration of products at lower price points.

**Decision: Retain Price outliers.**

These values were not removed because there was insufficient evidence that they represented data-entry errors. Statistical unusualness alone was not considered sufficient justification for deleting valid transactions.

### Overall Outlier Decision

No IQR-detected outliers were removed or capped. The outlier analysis was used as a diagnostic tool to identify unusual values, followed by contextual inspection before making a cleaning decision.

In [24]:
# Recording outlier detection results


quantity_outliers_detected = quantity_outlier_mask.sum()
price_outliers_detected = price_outlier_mask.sum()

quantity_outliers_removed = 0
price_outliers_removed = 0

print("OUTLIER SUMMARY")
print("-" * 50)
print("Quantity outliers detected:", quantity_outliers_detected)
print("Quantity outliers removed:", quantity_outliers_removed)
print("Price outliers detected:", price_outliers_detected)
print("Price outliers removed:", price_outliers_removed)

OUTLIER SUMMARY
--------------------------------------------------
Quantity outliers detected: 27342
Quantity outliers removed: 0
Price outliers detected: 34703
Price outliers removed: 0


In [25]:

# Finalizing Customer ID as a string identifier


df["Customer ID"] = df["Customer ID"].astype("string")

print(df.dtypes)

Invoice        string[python]
StockCode      string[python]
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID    string[python]
Country        string[python]
dtype: object


In [26]:

# Post-Cleaning Validation

print("POST-CLEANING VALIDATION")
print("=" * 60)

# 1. Dataset dimensions
print("\nDATASET")
print("-" * 60)
print("Rows:", len(df))
print("Columns:", len(df.columns))


# 2. Missing values
print("\nMISSING VALUES")
print("-" * 60)

missing_after = df.isnull().sum()

display(
    pd.DataFrame({
        "Missing Values": missing_after,
        "Missing %": (missing_after / len(df) * 100).round(2)
    })
)


# 3. Duplicate rows
print("\nDUPLICATES")
print("-" * 60)
print("Duplicate rows:", df.duplicated().sum())


# 4. Data types
print("\nDATA TYPES")
print("-" * 60)
print(df.dtypes)


# 5. Suspicious numeric values
print("\nNUMERIC VALIDATION")
print("-" * 60)
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Zero Quantity:", (df["Quantity"] == 0).sum())
print("Negative Price:", (df["Price"] < 0).sum())
print("Zero Price:", (df["Price"] == 0).sum())


# 6. Date validation
print("\nDATE VALIDATION")
print("-" * 60)
print("Minimum date:", df["InvoiceDate"].min())
print("Maximum date:", df["InvoiceDate"].max())


# 7. Text whitespace validation
print("\nTEXT FORMATTING VALIDATION")
print("-" * 60)

for col in ["Invoice", "StockCode", "Description", "Country"]:
    whitespace_count = (
        df[col].str.strip().ne(df[col])
    ).sum()

    print(f"{col}: {whitespace_count}")

POST-CLEANING VALIDATION

DATASET
------------------------------------------------------------
Rows: 410763
Columns: 8

MISSING VALUES
------------------------------------------------------------


,Missing Values,Missing %
Invoice,0,0.0
StockCode,0,0.0
Description,0,0.0
Quantity,0,0.0
InvoiceDate,0,0.0
Price,0,0.0
Customer ID,0,0.0
Country,0,0.0



DUPLICATES
------------------------------------------------------------
Duplicate rows: 0

DATA TYPES
------------------------------------------------------------
Invoice        string[python]
StockCode      string[python]
Description    string[python]
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID    string[python]
Country        string[python]
dtype: object

NUMERIC VALIDATION
------------------------------------------------------------
Negative Quantity: 9816
Zero Quantity: 0
Negative Price: 0
Zero Price: 31

DATE VALIDATION
------------------------------------------------------------
Minimum date: 2009-12-01 07:45:00
Maximum date: 2010-12-09 20:01:00

TEXT FORMATTING VALIDATION
------------------------------------------------------------
Invoice: 0
StockCode: 0
Description: 0
Country: 0


In [27]:

# Before vs After Cleaning Summary


rows_after = len(df)
missing_after_total = df.isnull().sum().sum()
duplicate_after = df.duplicated().sum()

summary = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Total Missing Values",
        "Duplicate Rows",
        "Quantity Outliers Detected",
        "Quantity Outliers Removed",
        "Price Outliers Detected",
        "Price Outliers Removed"
    ],
    "Before Cleaning": [
        rows_before,
        missing_before,
        duplicate_before,
        "-",
        "-",
        "-",
        "-"
    ],
    "After Cleaning": [
        rows_after,
        missing_after_total,
        duplicate_after,
        quantity_outliers_detected,
        quantity_outliers_removed,
        price_outliers_detected,
        price_outliers_removed
    ]
})

display(summary)

,Metric,Before Cleaning,After Cleaning
0,Row Count,525461,410763
1,Total Missing Values,110855,0
2,Duplicate Rows,6865,0
3,Quantity Outliers Detected,-,27342
4,Quantity Outliers Removed,-,0
5,Price Outliers Detected,-,34703
6,Price Outliers Removed,-,0


In [28]:

# Data Type Accuracy Check


expected_dtypes = {
    "Invoice": "string",
    "StockCode": "string",
    "Description": "string",
    "Quantity": "int64",
    "InvoiceDate": "datetime64[ns]",
    "Price": "float64",
    "Customer ID": "string",
    "Country": "string"
}

dtype_check = pd.DataFrame({
    "Column": df.columns,
    "Actual Type": [
        str(df[col].dtype) for col in df.columns
    ],
    "Expected Type": [
        expected_dtypes[col] for col in df.columns
    ]
})

dtype_check["Correct"] = (
    dtype_check["Actual Type"] ==
    dtype_check["Expected Type"]
)

display(dtype_check)

print(
    "Correct dtypes:",
    dtype_check["Correct"].sum(),
    "of",
    len(dtype_check)
)

,Column,Actual Type,Expected Type,Correct
0,Invoice,string,string,True
1,StockCode,string,string,True
2,Description,string,string,True
3,Quantity,int64,int64,True
4,InvoiceDate,datetime64[ns],datetime64[ns],True
5,Price,float64,float64,True
6,Customer ID,string,string,True
7,Country,string,string,True


Correct dtypes: 8 of 8


In [29]:

# Final Clean Dataset Preview


print("Final dataset shape:", df.shape)

display(df.head(10))

display(df.sample(10, random_state=42))

Final dataset shape: (410763, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
138436,502633,84676,BATH DUCK WATERING CAN,6,2010-03-25 16:19:00,2.95,12510,Spain
502522,536262,21890,S/6 WOODEN SKITTLES IN COTTON BAG,2,2010-11-30 15:33:00,2.95,14085,United Kingdom
47902,493860,40018F,CHERRY DESIGN PAPERLANTERNS,12,2010-01-07 16:19:00,1.95,13767,United Kingdom
265500,515019,22320,BIRDS MOBILE VINTAGE DESIGN,3,2010-07-08 09:00:00,5.95,12712,Germany
104263,499377,21274,EMBOSSED HEART 3 DRAWER SHELF,1,2010-02-26 11:59:00,12.75,14177,United Kingdom
375650,525711,21212,PACK OF 72 RETROSPOT CAKE CASES,10,2010-10-06 14:12:00,0.55,14085,United Kingdom
397678,527428,20974,12 PENCILS SMALL TUBE SKULL,2,2010-10-17 16:17:00,0.65,16283,United Kingdom
229740,511611,22274,FELTCRAFT DOLL EMILY,1,2010-06-09 11:28:00,2.95,15352,United Kingdom
137601,502599,85071B,RED CHARLIE+LOLA PERSONAL DOORSIGN,2,2010-03-25 13:16:00,2.95,17301,United Kingdom
188927,507359,22521,CHILDS GARDEN TROWEL PINK,12,2010-05-07 15:33:00,0.85,17028,United Kingdom


In [30]:

# Saving Cleaned Dataset


output_file = "cleaned_online_retail.csv"

df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")

Cleaned dataset saved as: cleaned_online_retail.csv


# Conclusion

The dataset was systematically cleaned and transformed into an analysis-ready format.

The cleaning process addressed:

* Missing values through evidence-based recovery and row deletion where values could not be reliably recovered.
* Exact duplicate transactions through duplicate detection and removal.
* Inconsistent text formatting through whitespace standardization.
* Incorrect data types, including conversion of Customer ID from a numeric representation to a string identifier.
* Unusual numeric values through IQR-based outlier detection and contextual inspection.
* Negative-price records associated with "Adjust bad debt" entries were identified during the initial quality assessment and did not remain in the final dataset after missing-value filtering.

The IQR analysis identified 27,342 potential Quantity outliers and 34,703 potential Price outliers. These were retained after contextual inspection because many represented legitimate business activity such as product returns, cancellations, bulk purchases, or valid higher-priced products.

The final dataset contains 410,763 rows and 8 columns, with no missing values or duplicate rows. All eight columns were validated against their expected data types, and no unnecessary whitespace remained in the standardized text fields.

The cleaned dataset was exported as `cleaned_online_retail.csv`.

This process demonstrates that effective data cleaning is not simply about removing unusual values; it requires understanding the meaning and business context of the data before making cleaning decisions.
